# Merging canvases

Normally each person runs their own `Canvas` on their own port. This notebook
launches two such canvases (as if they were two different people / machines)
and then a **merge host** that composites both onto one new port — the union of
every panel, on a single surface.

The key property: interacting with a panel in the merged view runs that panel's
logic back in the process that created it. Here both source canvases live in
this one kernel for demonstration, but they could just as easily be on two
laptops — the merge host only needs to reach their ports.

Because a notebook kernel stays alive, the source canvases keep serving their
panels for as long as this kernel runs — which is exactly what the merge host
needs to route interactions back to them.

In [1]:
import pycanvas

## "Sarah's" canvas, on port 8001

Give the panels explicit positions. A position-less panel has no coordinate to
preserve (each source's browser auto-places it, and that spot is never reported
back), so the merged view would just auto-cascade it. With explicit `x`/`y`,
each panel appears in the merged view exactly where its owner placed it.

In [2]:
sarah = pycanvas.Canvas()
s_slider = sarah.insert(
    pycanvas.Slider(name="sarah_dial", min=0, max=100, default=20), x=120, y=240
)


@s_slider.on_change
def on_sarah(value):
    print(f"[sarah's process] dial -> {value}")


sarah.serve(port=8001, open_browser=True, block=False)

PyCanvas serving at http://127.0.0.1:8001  (Ctrl+C to stop)


## "Josef's" canvas, on port 8002

In [3]:
josef = pycanvas.Canvas()
j_slider = josef.insert(
    pycanvas.Slider(name="josef_dial", min=0, max=100, default=80), x=120, y=100
)


@j_slider.on_change
def on_josef(value):
    print(f"[josef's process] dial -> {value}")


josef.serve(port=8002, open_browser=True, block=False)

PyCanvas serving at http://127.0.0.1:8002  (Ctrl+C to stop)


## The merge host: unify both onto port 8080

By default the canvases are overlaid with their real coordinates preserved.
Pass `region_width=...` (e.g. `pycanvas.Merge([8001, 8002], region_width=1500)`)
to instead spread each source into its own side-by-side region.

`serve(block=False)` returns immediately so the kernel stays interactive. Open the
printed URL: you'll see **both** sliders on one canvas. Drag either one — its
`on_change` fires in its own canvas (watch the printed `[sarah's process]` /
`[josef's process]` lines).

In [4]:
merged = pycanvas.Merge([8001, 8002]).serve(port=8080, open_browser=True, block=False, host="")
print("Merged canvas at http://127.0.0.1:8080")

PyCanvas serving  (Ctrl+C to stop):
  local:   http://127.0.0.1:8080
  network: http://192.168.50.76:8080   <- open this on another device on the same Wi-Fi
Merged canvas at http://127.0.0.1:8080


## Merging canvases from other networks (tunnels)

Above, every source is on `localhost`, so the merge host reaches them by port
(`Merge([8001, 8002])`). But the sources don't have to be local — or even on the
same network. If a collaborator exposes their canvas with a **tunnel**
(`their_canvas.serve(tunnel=True)`, see [`public_tunnel.py`](public_tunnel.py)),
they get a public `https://…` URL, and you can merge it by passing that URL as a
source:

```python
merged = pycanvas.Merge([
    "https://sarah-xxxx.trycloudflare.com",   # Sarah, tunneled from another network
    "https://josef-yyyy.loca.lt",             # Josef, tunneled from another network
    ":8003",                                   # someone local, still by port
]).serve(port=8080, block=False)
```

`http(s)://` source URLs are rewritten to `ws(s)://…/ws` automatically; bare
ports / `host:port` keep using `ws://` as before. Interactions still route back
to whichever process owns the panel — now across the internet rather than the
LAN. The merge host itself can be tunneled too, so viewers anywhere open the
merged view:

```python
merged = pycanvas.Merge([...]).serve(port=8080, tunnel=True, block=False)
```


## Drive a panel from Python

Each source is still a normal `Canvas` you can control from this kernel. Setting
a slider's value in Python pushes to every view — including the merged one.

In [ ]:
s_slider.update(75)   # moves sarah_dial in both Sarah's canvas and the merged view

## Shut everything down

Stop the merge host and both source canvases.

In [ ]:
merged.stop()
sarah.stop()
josef.stop()